In [ ]:
pip install langchain langchain-google-genai google-generativeai langchain_community faiss-cpu pypdf chromadb

In [ ]:

specs = {}
specs['GOOGLE_API_KEY'] = ''
specs['MODEL_NAME'] = "gemini-1.5-pro-001"
specs['TOKENS'] = 30
specs['EMBED_MODEL_NAME'] = "models/text-embedding-004"


In [12]:
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(model=specs['MODEL_NAME'],max_tokens=specs['TOKENS'],api_key=specs['GOOGLE_API_KEY'])




In [11]:
# llm.invoke("Explain a Large Language Model in one line?")
llm.invoke("What is national sport of canada")

AIMessage(content='The national sports of Canada are **lacrosse** (summer sport) and **ice hockey** (winter sport). \n\n* **Lacrosse**', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-1.5-pro-001', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-9ea143b3-ba29-472e-810e-40c08f4fe506-0', usage_metadata={'input_tokens': 7, 'output_tokens': 30, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}})

In [10]:
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=specs['MODEL_NAME'],api_key=specs['GOOGLE_API_KEY'],max_tokens=specs['TOKENS'])

llm([
    SystemMessage(content='You should act as an experienced chef and answer the question which is related to cooking and recipes.'),
    HumanMessage(content='I dont like tomatoes, what else can make me a sandwich? Give a 2 line recipie.')
])

AIMessage(content='Skip the tomato worry!  Slather toasted bread with creamy avocado, then pile high with crispy bacon and crunchy sprouts. ', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-1.5-pro-001', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-6792458b-6ff9-4d27-bae0-141cc131ab2a-0', usage_metadata={'input_tokens': 41, 'output_tokens': 24, 'total_tokens': 65, 'input_token_details': {'cache_read': 0}})

In [13]:
from langchain import PromptTemplate

template = """
I want to be {career_option} in future. What subjects should I start studying?
Respond in 1-2 short sentence
"""

# Creating a template from the above prompt
prompt = PromptTemplate(
    input_variables=["career_option"],
    template=template
)

final_prompt = prompt.format(career_option='Machine Learning Engineer')

llm(final_prompt)

'Focus on building a strong foundation in mathematics (calculus, linear algebra, statistics, probability) and computer science (programming, algorithms, data structures). '

In [32]:
from langchain.prompts.example_selector import SemanticSimilarityExampleSelector
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.prompts import FewShotPromptTemplate
from langchain_community.vectorstores import FAISS
prompt = PromptTemplate(
    input_variables=["input", "output"],
    template="Example Input: {input}\nExample Output: {output}",
)
# Examples of job roles and respective job titles
examples = [
    {"input": "software engineer", "output": "software development"},
    {"input": "accountant", "output": "accounting"},
    {"input": "teacher", "output": "education"},
    {"input": "doctor", "output": "medicine"},
    {"input": "architect", "output": "architecture"},
    {"input": "lawyer", "output": "law"},
]
selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    GoogleGenerativeAIEmbeddings(model=specs['EMBED_MODEL_NAME'], google_api_key=specs['GOOGLE_API_KEY']),
    FAISS,
    k=2
)


In [33]:
similar_prompt = FewShotPromptTemplate(example_selector=selector, 
									   example_prompt=prompt,  
    								   prefix="Give the job title their job role is ", 
    								   suffix="Input: {job_title}\nOutput:",
    								   input_variables=["job_title"] 
    								   )

In [ ]:
print(similar_prompt.format(job_title='nurse'))


Give the job title their job role is 

Example Input: doctor
Example Output: medicine

Example Input: accountant
Example Output: accounting

Input: nurse
Output:


In [35]:
llm(similar_prompt.format(job_title='nurse'))

'Output: nursing '

File Loading & Document Analyzing

In [23]:

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import PythonCodeTextSplitter

# Splitting text into chunks, entire text shouldn't be given at once to the model.
# It should be splitted into chunks
text_splitter = PythonCodeTextSplitter(chunk_size = 1000, chunk_overlap=300)

loader = PyPDFLoader('sample.pdf')  
data = loader.load_and_split(text_splitter=text_splitter)
print(data[0].page_content)
print (f"Found {len(data)} comments")

Napoleon  Bonaparte  was  born  on  August  15,  1769,  in  Corsica.  He  graduated  
from
 
the
 
École
 
Militaire
 
in
 
Paris
 
in
 
1785
 
as
 
an
 
artillery
 
officer.
 
His
 
military
 
career
 
took
 
off
 
in
 
1793
 
when
 
he
 
gained
 
recognition
 
for
 
recapturing
 
Toulon
 
from
 
the
 
British.
 
In
 
1795,
 
he
 
suppressed
 
a
 
royalist
 
uprising
 
in
 
Paris,
 
earning
 
him
 
command
 
of
 
the
 
French
 
Army
 
of
 
Italy.
 
Between
 
1796
 
and
 
1797,
 
he
 
led
 
successful
 
campaigns
 
in
 
Italy,
 
defeating
 
Austrian
 
forces.
 
In
 
1798,
 
he
 
invaded
 
Egypt
 
to
 
weaken
 
British
 
influence
 
but
 
suffered
 
a
 
naval
 
defeat
 
at
 
the
 
Battle
 
of
 
the
 
Nile.
 
In
 
1799,
 
he
 
overthrew
 
the
 
French
 
government
 
in
 
the
 
Coup
 
of
 
18
 
Brumaire
 
and
 
became
 
First
 
Consul
 
of
 
France.
 
By
 
1804,
 
he
 
had
 
declared
 
himself
 
Emperor
 
of
 
France
 
after
 
a
 
national
 
referendum.
 
One
 
of
 
his
 
greatest
Found 3

Storing the Documents in VectorStores (Chrome built-in memory store)

In [24]:
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

chroma = Chroma.from_documents(
    documents=data, 
    embedding=GoogleGenerativeAIEmbeddings(model=specs['EMBED_MODEL_NAME'], google_api_key=specs['GOOGLE_API_KEY']),    
    collection_name='sample'
)   

In [27]:
similarities = chroma.similarity_search("When did he invaded egypt")

similarities[0].page_content


'Napoleon  Bonaparte  was  born  on  August  15,  1769,  in  Corsica.  He  graduated  \nfrom\n \nthe\n \nÉcole\n \nMilitaire\n \nin\n \nParis\n \nin\n \n1785\n \nas\n \nan\n \nartillery\n \nofficer.\n \nHis\n \nmilitary\n \ncareer\n \ntook\n \noff\n \nin\n \n1793\n \nwhen\n \nhe\n \ngained\n \nrecognition\n \nfor\n \nrecapturing\n \nToulon\n \nfrom\n \nthe\n \nBritish.\n \nIn\n \n1795,\n \nhe\n \nsuppressed\n \na\n \nroyalist\n \nuprising\n \nin\n \nParis,\n \nearning\n \nhim\n \ncommand\n \nof\n \nthe\n \nFrench\n \nArmy\n \nof\n \nItaly.\n \nBetween\n \n1796\n \nand\n \n1797,\n \nhe\n \nled\n \nsuccessful\n \ncampaigns\n \nin\n \nItaly,\n \ndefeating\n \nAustrian\n \nforces.\n \nIn\n \n1798,\n \nhe\n \ninvaded\n \nEgypt\n \nto\n \nweaken\n \nBritish\n \ninfluence\n \nbut\n \nsuffered\n \na\n \nnaval\n \ndefeat\n \nat\n \nthe\n \nBattle\n \nof\n \nthe\n \nNile.\n \nIn\n \n1799,\n \nhe\n \noverthrew\n \nthe\n \nFrench\n \ngovernment\n \nin\n \nthe\n \nCoup\n \nof\n \n18\n \nBrumaire\n 